In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create project structure
!mkdir -p /content/drive/MyDrive/dataSynthesizer/{raw_data,processed_data,models,outputs,logs}

In [ ]:
!pip install -q pandas openpyxl faker transformers[torch] datasets accelerate scikit-learn scipy python-dateutil tqdm
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install ctgan sdv
!pip install -q sdv

In [ ]:
# preprocess_finance.py
import os
import glob
import json
import hashlib
from pathlib import Path
from faker import Faker
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm import tqdm

fake = Faker()
FOLDER = "/content/drive/MyDrive/dataSynthesizer/datasets"
OUT = "/content/drive/MyDrive/dataSynthesizer/processed_data"
os.makedirs(OUT, exist_ok=True)

# Column normalization map
STANDARD_MAP = {
    "txn_id": ["transactionid", "txnid", "transaction_id", "id"],
    "date": ["date", "transaction_date", "txn_date", "timestamp"],
    "amount": ["amount", "amt", "transaction_amount", "value"],
    "category": ["category", "merchant_category", "cat"],
    "merchant": ["merchant", "vendor", "payee"],
    "mode": ["mode", "payment_mode", "payment_method"],
    "balance": ["balance", "account_balance", "bal"],
    "customer_id": ["customerid", "customer_id", "cust_id", "user_id"],
    "name": ["name", "customer_name", "full_name"],
    "email": ["email", "email_address"],
    "phone": ["phone", "phone_number", "mobile"]
}

def normalize_colname(c):
    c2 = str(c).lower().strip().replace(" ", "_")
    for std, variants in STANDARD_MAP.items():
        if c2 == std or c2 in variants:
            return std
    # fuzzy fallback: keep alphanumeric underscores
    return "".join(ch for ch in c2 if ch.isalnum() or ch == "_")

def anonymize_pii(df):
    # Replace name/email/phone with fake values if present
    if "name" in df.columns:
        df["name"] = [fake.name() for _ in range(len(df))]
    if "email" in df.columns:
        df["email"] = [fake.email() for _ in range(len(df))]
    if "phone" in df.columns:
        df["phone"] = [fake.phone_number() for _ in range(len(df))]
    # Hash customer_id or txn_id to avoid original IDs leaking
    for col in ("customer_id", "txn_id", "transactionid", "id"):
        if col in df.columns:
            df[col] = df[col].astype(str).apply(
                lambda x: hashlib.sha256(x.encode()).hexdigest()[:12]
            )
    return df

def standardize_categories(series):
    # simple normalization: lowercase and title-case common bins
    def norm(x):
        try:
            s = str(x).strip().lower()
            if s in ("grocery", "groceries", "supermarket"):
                return "Groceries"
            if s in ("food", "restaurant", "dining"):
                return "Food & Dining"
            if "salary" in s or "pay" in s:
                return "Salary"
            if "fuel" in s or "petrol" in s:
                return "Fuel"
            if s in ("", "nan", "none"):
                return "Other"
            return s.title()
        except:
            return "Other"
    return series.apply(norm)

def process_file(path):
    name = Path(path).stem
    print(f"Processing {path} → domain: {name}")
    # read robustly
    try:
        if path.lower().endswith((".xls", ".xlsx")):
            df = pd.read_excel(path, engine="openpyxl")
        else:
            df = pd.read_csv(path)
    except Exception as e:
        print("Read failed:", e)
        return None, None

    # Normalize column names
    rename_map = {c: normalize_colname(c) for c in df.columns}
    df = df.rename(columns=rename_map)

    # Standardize common columns if present
    if "category" in df.columns:
        df["category"] = standardize_categories(df["category"])
    if "date" in df.columns:
        # try to parse dates
        df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.strftime("%Y-%m-%d")
    if "amount" in df.columns:
        df["amount"] = pd.to_numeric(df["amount"], errors="coerce").fillna(0.0)

    # Anonymize PII
    df = anonymize_pii(df)

    # Basic cleanup: drop fully-empty cols
    df = df.dropna(axis=1, how="all")
    return name, df

def make_prompt_output_pairs(df, domain, out_jsonl_path, sample_limit=None):
    """
    Create JSONL lines with {"prompt": "...", "output": "..."} where output is JSON string.
    """
    rows = df.to_dict(orient="records")
    if sample_limit:
        rows = rows[:sample_limit]

    def safe_json(v):
        if pd.isna(v):
            return None
        if isinstance(v, (pd.Timestamp, )):
            return v.strftime("%Y-%m-%d")
        return v

    with open(out_jsonl_path, "w", encoding="utf-8") as f:
        for r in rows:
            fields = list(r.keys())
            prompt = f"Generate a realistic {domain} record with fields: {', '.join(fields)}."
            output_obj = {k: safe_json(v) for k, v in r.items()}
            line = {"prompt": prompt, "output": json.dumps(output_obj, ensure_ascii=False)}
            f.write(json.dumps(line, ensure_ascii=False) + "\n")


def main():
    files = glob.glob(os.path.join(FOLDER, "*"))
    per_domain = {}
    for p in tqdm(files):
        domain, df = process_file(p)
        if domain is None or df is None:
            continue
        per_domain[domain] = df
        # save cleaned CSV
        csv_out = os.path.join(OUT, f"{domain}_clean.csv")
        df.to_csv(csv_out, index=False)
        print("Saved", csv_out)
        # create prompt-output pairs
        jsonl_out = os.path.join(OUT, f"{domain}_prompts.jsonl")
        make_prompt_output_pairs(df, domain, jsonl_out, sample_limit=50000)  # cap to 50k
        print("Saved prompts", jsonl_out)
        # create train/val/test split for tabular model
        train, test = train_test_split(df, test_size=0.2, random_state=42)
        train.to_csv(os.path.join(OUT, f"{domain}_train.csv"), index=False)
        test.to_csv(os.path.join(OUT, f"{domain}_test.csv"), index=False)
    print("Preprocessing complete. Files in", OUT)

if __name__ == "__main__":
    main()
